<a href="https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I am choosing **Refresh / Content Opportunity Scoring** and framing it as a **ranking task**. The goal is to help an SEO analyst decide which existing webpages deserve review first when editorial time is limited. The system would rank eligible content items using observable search-performance and content signals. A reviewer could then investigate the highest-priority pages and decide whether to refresh, expand, monitor, consolidate, or leave them unchanged. Ranking is appropriate because the main decision is not simply whether a page is good or bad; it is which pages should receive attention first. I will compare a learned ranking with a transparent rule-based baseline before claiming that machine learning adds value.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

task_type = "Ranking / scoring"
lane = "Refresh / Content Opportunity Scoring"
decision = "Which pages should an SEO analyst review first?"

print("Lane:", lane)
print("ML task type:", task_type)
print("Decision:", decision)

Lane: Refresh / Content Opportunity Scoring
ML task type: Ranking / scoring
Decision: Which pages should an SEO analyst review first?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the initial starter-data exercise, I will use `is_declining_label`, equivalent to `trend_direction == "down"`, as a **proxy label** for an observed downward trend. This is not a future-outcome label: it is derived from the current performance window. It can help demonstrate the ranking workflow, but it cannot establish whether a page will decline in the future or whether a content refresh would improve its performance. I will exclude `trend_direction`, `trend_pct`, and `is_declining_label` from the input features when modeling this proxy, because those fields directly define the label or reveal its answer. A stronger future-facing capstone would construct a separate target from a later time window.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    url = (
        "https://raw.githubusercontent.com/"
        "SoyebAhmed5/flyrank-internship-ml/main/"
        "data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(url)

required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction",
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

df["target_current_down"] = (
    df["trend_direction"].eq("down").astype("int8")
)

print("Proxy target: 1 = downward trend; 0 = otherwise")
print(df["target_current_down"].value_counts(dropna=False))

Proxy target: 1 = downward trend; 0 = otherwise
target_current_down
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric will be **Precision@50**. It measures the proportion of the top 50 recommended pages that have the positive proxy label. For example, Precision@50 of 0.60 would mean that 30 of the 50 highest-ranked pages have the observed downward-trend label. I will compare the model against a simple rule-based baseline using the same held-out clients. I will consider the model useful only if it improves ranking quality without introducing leakage and provides actionable recommendations. Because the label represents a current-window trend, this metric evaluates agreement with that proxy, not successful future recovery after a refresh.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


K = 50

print("Primary metric: Precision@50")
print("Evaluation rule: compare model and baseline on the same held-out clients.")
print("Positive outcome: target_current_down == 1")
print(
    "Precision@50 = positive labels among top K "
    "/ number of evaluated items in top K"
)

Primary metric: Precision@50
Evaluation rule: compare model and baseline on the same held-out clients.
Positive outcome: target_current_down == 1
Precision@50 = positive labels among top K / number of evaluated items in top K


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is **one pseudonymized content item**. Each row represents one content item, not one client or one day. I will inspect the eligible starter-data slice, keeping items with positive 90-day impressions and at least 90 days of content age, and deduplicating by `content_id`. The dataframe will show the content identifier, client grouping identifier, selected observable signals, and proxy target. IDs will be used for grouping and validation, not as predictive features.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Apply the starter pipeline's documented eligibility conditions.
eligible = df.loc[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].copy()

eligible = eligible.drop_duplicates(
    subset="content_id",
    keep="first",
)

# Add useful fields only when they exist in this dataset.
display_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction",
    "target_current_down",
]

optional_columns = [
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "word_count",
]

display_columns += [
    col for col in optional_columns
    if col in eligible.columns
]

unit_df = eligible[display_columns].copy()

print("Unit of analysis: one content item per row")
print("Eligible rows:", len(unit_df))
print("Unique content items:", unit_df["content_id"].nunique())

display(unit_df.head(10))


Unit of analysis: one content item per row
Eligible rows: 30000
Unique content items: 30000


,content_id,client_id,impressions_90d,content_age_days,trend_direction,target_current_down,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,word_count
0,content_304f48230142,client_f369cb89fc,3803,187,down,1,29,17,0.76,10.6,5.88,3221.0
1,content_a1fb4e703a9e,client_4e07408562,15320,445,down,1,7,9,0.05,20.3,0.00,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,12581,141,down,1,11,11,0.09,36.5,0.00,3515.0
3,content_331d6c4de07b,client_19581e27de,11751,463,stable,0,58,78,0.49,6.2,1.28,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,19140,263,down,1,24,145,0.13,44.0,0.00,2803.0
5,content_d4084a4bc775,client_f369cb89fc,3970,147,down,1,1,5,0.03,8.5,0.00,3080.0
6,content_9a34b442b552,client_8722616204,20,90,down,1,0,1,0.00,7.0,0.00,3059.0
7,content_a63219c6e95a,client_19581e27de,1724,445,stable,0,1,28,0.06,21.2,3.57,NaN
8,content_5e6c160719bc,client_6208ef0f77,32574,90,down,1,29,68,0.09,46.0,5.88,3807.0
9,content_c27558df2b0c,client_19581e27de,1240,257,down,1,2,3,0.16,4.9,0.00,NaN


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule is useful as a baseline because it is simple, transparent, and easy to explain. However, content-review opportunities may depend on several interacting signals, such as visibility, content age, search position, click-through rate, and engagement. A single threshold may miss combinations of signals that deserve attention. A learned model may rank pages more effectively by combining these signals, but that benefit must be demonstrated rather than assumed. I will compare the learned approach with a fixed-rule baseline using the same held-out clients and Precision@50. I will prefer the simpler rule if the model does not provide a meaningful improvement. Any resulting queue will support human review, not automatically authorize content changes.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Define the comparison before training any model.
comparison_plan = pd.DataFrame({
    "Approach": [
        "Fixed-rule baseline",
        "Learned ranking model",
    ],
    "Purpose": [
        "Provide a transparent reference",
        "Test whether combining signals improves ranking",
    ],
    "Evaluation": [
        "Precision@50 on held-out clients",
        "Precision@50 on the same held-out clients",
    ],
})

display(comparison_plan)


,Approach,Purpose,Evaluation
0,Fixed-rule baseline,Provide a transparent reference,Precision@50 on held-out clients
1,Learned ranking model,Test whether combining signals improves ranking,Precision@50 on the same held-out clients


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.